# Introduction

`keecas` is a module for performing symbolic and units-aware calculations. It has been developed to be used mainly in a jupyter notebook's interactive environments, which then could be rendered as pdf by [quarto](https://quarto.org/).

`keecas` leverage some well known python modules, wrapping some of their functions with sensible defaults, aiming to reduce boiler plate code and be quick to use. Some of the modules used are:

`sympy`
: for all symbolic expression and computation

`pint`
: convenient unit registry; `pint` quantities can be converted in `sympy.physics.units` directly

`pipe`
: some of the most used `sympy` functions are wrapped in `pipe` object, so that they can be used in a sequential fashion



## Quick Start

Let's start with a bare minimum example: let's calculate the maximum bending moment ($M_{Sd}$) for a beam of span $l$, simply supported at the end (pinned), uniformly loaded ($q$).

In [34]:
# notable imports
from keecas import (
    show_eqn, # main function to generate LaTeX amsmath block
    check, # main function for verification
    symbols, # to create symbols (from sympy)
    u, # unit registry from pint
    pc, # pipe command namespace
    config, # configuration options
)

# or import everything from keecas (more function are imported this way)
# from keecas import *

In [35]:
# define the symbols with symbols() from `sympy`
q, l, M_Sd = symbols(r'q, l, M_{Sd}')

# define the parameters with units support provided by `pint` (`u` is the unit registry)
_p = {
    q: 5*u('kN/m'),
    l: 400*u.cm,
}

# define the symbolic expression
_e = {
    M_Sd: "q*l^2/8" # the expression is written as a simple string
    | pc.parse_expr, # function that will parse the expression (from `sympy`)
}

# evaluate the expression in `_e`
_v = {
    k: v
    | pc.subs(_e|_p) # subistitute the symbol contained in the joined dict _e|_p (from `sympy`)
    | pc.convert_to([u.kN, u.m]) # convert the expression (from `sympy`) to the preferred units (list of `pint` units)
    | pc.N # elavuate the expression as decimal (from `sympy`)
    for k,v in _e.items() # get all the expression contained in _e
}

# add some description
_d = {
    M_Sd: "max bending moment",
    q: "uniform load",
    l: "span of the beam",
}

# create the latex expression (amsmath) to be displayed as Markdown object
show_eqn(
    [
        _p|_e,  # first dict: keys will be shown as first column, values as second column
        _v,     # second dict: only values will be shown third column, only if there is a matching key
        _d,     # same as above +1 column
    ], # list of dict to be shown
    float_format='{:.2f}', # precision of floats
)

<IPython.core.display.Latex object>

We can have a look at the latex code by passing `debug=True` to the function or setting globally `config.display.debug=True`

In [36]:
## Global configuration
# config.display.debug=True

show_eqn(
    [_p|_e, _v, _d], # list of dict to be shown
    float_format=['{:.2f}', '.3f'], # precision of floats for each column (last value repeats for following columns)
    debug=True, # set debug True to print LaTeX code
    environment='align',
)

\begin{align}
q & = 5.000{\,}\dfrac{\text{kN}}{\text{m}} &   & \quad\text{uniform load}  \\[8pt]
 l & = 400{\,}\text{cm} &   & \quad\text{span of the beam}  \\[8pt]
 M_{Sd} & = \dfrac{q{\,}l^{2}}{8} & = 10.000{\,}\text{kN}{\,}\text{m} & \quad\text{max bending moment} 
\end{align}


<IPython.core.display.Latex object>

## A more structured example: Simple Beam

In the previous example all the `dict` were prepended by `_`: that because those dict are not really meant to be preserved further than the cell they are defined. They are meant to only collect the expressions/parameters of the cell to be displayed, and then discarded. To preserve all the expression/parameters of the notebook wwe will initialize a named dict.


In [37]:
# this dict will act as namespace for all the expressions/parameters defined in this file (could be used in other notebook)
simple_beam = {
    'parameters' : (params := {}), # params is the dict that will collect all the params of this notebook
    'expressions': (eqn := {}), # eqn is the dict that will collect all the eqn of this notebook
}

We will calculate the notable values for a simple supported beam with a uniform load in ULS condition for bending and shear, and SLS condition for deflection.

The beam is characterized by this parameters:

In [38]:
l, b_i = symbols(r"l, b_{i}")

_p = {
    l: u('6 m'), # the unitregistry can interpret a string
    b_i: 3*u.m,
}
params.update(_p) # save the parameters in the notebook dict

_d = {
    l: "span of the beam",
    b_i: "width of influence",
}

show_eqn([_p, _d], environment="cases")

<IPython.core.display.Latex object>

### Beam properties

For this example an `IPE270` is considered, characterized by the following properties:

In [39]:
# comma needs escaping
E_s, J_yy, W_pl_yy, A_v = symbols(r"E_{s}, J_{yy}, W_{pl\,yy}, A_{v}") 
b, h, t_f, t_w, r, A = symbols(r"b, h, t_{f}, t_{w}, r, A")


_p = {
    E_s: 210000*u.MPa,
    J_yy: 5790*u.cm**4,
    W_pl_yy: u('484 cm^3'),
    A: 45.94*u.cm**2,
    b: 135*u.mm,
    h: 270*u.mm,
    t_w: 6.6*u.mm,
    t_f: 10.2*u.mm,
    r: 15*u.mm,
}
params.update(_p)

_d = {
    E_s: "elastic modulus",
    J_yy: "moment of inertia",
    W_pl_yy: "plastic section modulus",
    A: "cross section area",
    b: "beam width",
    h: "beam height",
    t_w: "web thickness",
    t_f: "flange thickness",
    r: "flange radius",    
}

# import the default wrap_column dispatch function that can be passed to col_wrap argument, to overwrite the default behavior
from keecas import wrap_column

show_eqn(
    [_p, _d], 
    col_wrap=[wrap_column]*2+['&'] # overwrite the default behavior to have the text in the third column left aligned
)

<IPython.core.display.Latex object>

`S275` is assumend as material, with the following characteristics:

In [40]:
f_sk, gamma_M0 = symbols(r"f_{sk}, \gamma_{M0}")

_p = {
    f_sk: 275*u.MPa,
    gamma_M0: 1.05,
}
params.update(_p)

_d = {
    f_sk: "characteristic strength",
    gamma_M0: "safety factor",
}

# text is left aligned without any col_wrap override
show_eqn([_p, _d])

<IPython.core.display.Latex object>

### Actions

Let's calculate the load applied to a simple beam.

#### Loads

In [41]:
G_1, G_2, Q_k = symbols(r"G_{1}, G_{2}, Q_{k}")


# define the loads in whatever units you want
_p = {
    G_1: 2.5 * u.kPa,
    G_2: 300 * u("daN/m^2"),
    Q_k: 4 * u.kN / u.m**2,
}
params.update(_p)  # save the parameters in the notebook dict

_d = {
    G_1: "permanent loads",
    G_2: "permanent non structural",
    Q_k: "live loads",
}

show_eqn(
    [_p, _d],
    col_wrap=[wrap_column, wrap_column, '&'] # We want the text in the third column to be left align, so another '&' is needed (the first 2 columns can be wrapped as usual)
    )

<IPython.core.display.Latex object>

#### Safety coefficients for loads

In [42]:
gamma_G1, gamma_G2, gamma_Qk = symbols(r'\gamma_{G_{1}}, \gamma_{G_{2}} , \gamma_{Q_{k}}')

_p = {
    gamma_G1: 1.3,
    gamma_G2: 1.5,
    gamma_Qk: 1.5,
}
params.update(_p)

# since _p is being redefined in this cell, it will only display the content of this cell
show_eqn(_p)

<IPython.core.display.Latex object>

#### Applied forces

The applied load are multiplied for the width of influence $b_i$ of the beam:

In [43]:
F_d, F_k = symbols(r"F_{d}, F_{k}")

_e = {
    F_k: "(G_1 + G_2 + Q_k)*b_i" | pc.parse_expr,
    F_d: "(gamma_G1*G_1+gamma_G2*G_2+gamma_Qk*Q_k)*b_i" | pc.parse_expr,
}
eqn.update(_e)  # save the expressions in the notebook dict

_v = {
    k: (
        v
        | pc.subs(eqn | params)
        | pc.convert_to([u.kN, u.m])
        | pc.N
    )
    for k, v in _e.items()
}

_d ={
    F_k: "(SLS)",
    F_d: "(ULS)",
}

show_eqn([_e, _v, _d], float_format='{:.2f}')

<IPython.core.display.Latex object>

### ULS: Bending Moment and Shear

#### Internal forces

We calculate the bending moment and shear in ULS condition:



In [44]:
M_Sd, V_Sd = symbols(r"M_{Sd}, V_{Sd}")

_e = {
    M_Sd: "F_d * l^2 / 8" | pc.parse_expr,
    V_Sd: "F_d * l / 2" | pc.parse_expr,
}
eqn.update(_e)  # save the expressions in the notebook dict

_v = {
    k: (
        v
        | pc.subs(eqn | params)
        | pc.convert_to([u.kN, u.m])
        | pc.N
        | pc.as_two_terms(as_mul=True) # this will display nicely the values and units in the show_eqn function
    )
    for k, v in _e.items()
}

# passing a dict as float_format we can specify specific format applied to the whole row (int are not affected)
_f = {
    M_Sd: '{:.3f}',
    V_Sd: '{:.0f}',
}

show_eqn(
    [_e, _v, _d],
    float_format=_f,
    )

<IPython.core.display.Latex object>

#### Verification

`keecas` provide the `check` function which will emit a LaTeX template based on True/False statements. It is usually combined with the `show_eqn` function.

We calculate the $M_{Rd}$ and $V_{Rd}$ in the following way

In [53]:
M_Rd, V_Rd = symbols(r"M_{Rd}, V_{Rd}")

_e = {
    M_Rd: "f_sk/gamma_M0 * W_pl_yy" | pc.parse_expr,
    V_Rd: "A_v*f_sk/(sqrt(3)*gamma_M0)" | pc.parse_expr,
    A_v: "A-2*b*t_f+(t_w+2*r)*t_f" | pc.parse_expr,
}
eqn.update(_e) 

_u ={
    M_Rd: [u.kN, u.m],
    V_Rd: [u.kN, u.m],
    A_v: u.cm, 
}

_v = {
    k: v | pc.subs(eqn | params) | pc.convert_to(_u[k]) | pc.N for k, v in _e.items()
}



show_eqn(
    [_e, _v, _d],
    float_format='{:.1f}',
)

<IPython.core.display.Latex object>

### SLS: deflection

We calculate the deflection of the beam with the following equation:

In [ ]:
f, E, J = symbols(r"f , E, J")

# since we want to collect display the expression as two separate terms (5/384) and (F_k * l^4 / (E *J)), we will wrap in `S()` (singleton from sympy);
# uncomment the line to see the different display results
_e = {
    # f : " (5/384) * (F_k * l^4 / (E *J))" | pc.parse_expr,
    f : " S(5/384) * S(F_k * l^4 / (E *J))" | pc.parse_expr,
    # f : " N(5/384) * S(F_k * l^4 / (E *J))" | pc.parse_expr, # th N() will evaluate the fraction to a decimal

}
eqn.update(_e)

show_eqn(_e)

<IPython.core.display.Latex object>

Assuming a `IPE270` beam, the deflection results

In [ ]:
_p = {
    E: 210000 * u("MPa"),
    J: 8356*u.cm**4,
}
params.update(_p)

_v = {
    k: (
        v
        | pc.subs(eqn | params)
        | pc.convert_to([u.kN, u.mm])
        | pc.N
    )
    for k, v in _e.items()
}

# We can evaluate a simple expression that has not its own symbol defined
__v = {
    k: k | pc.subs(eqn | params) | pc.convert_to([u.kN, u.mm]) | pc.N for k in [l/f]
}

_d = {
    l/f: "ratio span of the beam / deflection",
}

show_eqn([_p|_v|__v, _d], float_format='{:.2f}')

<IPython.core.display.Latex object>

## Verification

`keecas` provide the `check` function which will emit a LaTeX template based on True/False statements. It is usually combined with the `show_eqn` function.

### ULS

Assuming the `IPE270` 



In [ ]:
show_eqn(
    {
        'piecewise': '''Piecewise(
            (1, x<0), 
            (2, True),
        )
        ''' | pc.parse_expr,
    },
)

<IPython.core.display.Latex object>

# Labels

Various showcase of labeling generation:

In [ ]:
# setup
from keecas import generate_label, generate_unique_label

config.display.katex = True

J, E, a, b, c_0 = symbols(r"J, E, a, b, c_0")

# dictionary to be displayed by show_eqn
_p = {
    J: 123*u.cm**4,
    E: 456*u.MPa,
    a: "b+c_0/2" | pc.parse_expr,
}

## Manual assignment

In [ ]:
# when manually assigning the labels, they should be LaTeX safe
_l = {
    J: "eq-moment-of-inertia",
    E: "eq-modulus-of-elasticity",
    a: "eq-an-expression",
}

# pass _l to show_eqn
show_eqn(_p, label=_l)


<IPython.core.display.Latex object>

## Partial automatic generation

### Non unique labels

In [ ]:
# when manually assigning the labels, they should be LaTeX safe
_l = {
    J: "moment-of-inertia",
    E: "modulus-of-elasticity",
    a: "an-expression",
}


# generate the labels from the description
_l = generate_label(_l)
print(_l)

# pass _l to show_eqn
show_eqn(_p, label=_l)

{J: 'eq-moment-of-inertia', E: 'eq-modulus-of-elasticity', a: 'eq-an-expression'}


<IPython.core.display.Latex object>

### Unique labels

In [ ]:
_d = {
    J: "moment of inertia",
    E: "modulus of elasticity",
    a: "an expression",
}

# generate the labels from the description
_l = generate_label(_d, unique_id=True)
print(_l)

# pass _l to show_eqn
show_eqn(_p, label=_l)


{J: 'eq-534vb10v', E: 'eq-1sipf20t', a: 'eq-30phqdq8'}


<IPython.core.display.Latex object>

## Full automatic generation

In [ ]:
# pass _l to show_eqn
show_eqn([_p, _d], label=generate_unique_label)


<IPython.core.display.Latex object>